# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset, as described by its Croissant schema.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset's metadata and records using `mlcroissant`. This will initialize the dataset with its full schema and enable record access.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)
# Access metadata (as object, use .property or methods to access fields)
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")


## 2. Data Overview
Explore available record sets (tables), fields (columns), and their `@id`s. We'll enumerate all record sets, and for each, list its fields, columns, and their IDs as described in the Croissant schema.


In [ ]:
# List all available record sets, their fields and columns, always referencing by @id
print("Available Record Sets (tables):\n")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - {col.name} (@id: {col.id})")
    print()
if not record_sets:
    print("No record sets defined in this Croissant schema.\nCheck if the dataset includes files with records.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entities are referenced by their `@id` fields for consistency with the Croissant schema.


In [ ]:
# Gather all record set @id, e.g. record_sets[0].id, etc.
record_set_ids = [rs.id for rs in record_sets]
print("Record set @id values:\n", record_set_ids)
dataframes = {}
for rs in record_sets:
    print(f"Loading records from record set: {rs.name} (@id: {rs.id})...")
    try:
        records = list(dataset.records(record_set=rs.id))
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"  DataFrame shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}\n")
    except Exception as e:
        print(f"  Failed to load records: {e}\n")

# Example: Show first rows of the first available record set
if record_set_ids:
    example_id = record_set_ids[0]
    print(f"First records from record set {example_id}:")
    display(dataframes[example_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common EDA steps such as filtering, normalization, and grouping. To proceed, we pick a numeric field from the first record set for demonstration. All `@id` lookups are used for referencing fields/columns.


In [ ]:
# Choose a numeric field (by field @id) from the first record set, e.g. 'cr:coverage', 'cr:MW', etc.
numeric_field_id = None
group_field_id = None
if record_sets:
    rs = record_sets[0]
    # Try to select a numeric field by inspecting field data types
    for field in rs.fields:
        if field.data_type in ['Number', 'Float', 'Integer']:
            numeric_field_id = field.id
            break
    # As an example, pick a group by field (e.g., protein description or accession, if present)
    for field in rs.fields:
        if field.id != numeric_field_id:
            group_field_id = field.id
            break

    df = dataframes.get(rs.id)
    if numeric_field_id and numeric_field_id in df.columns:
        # Filtering
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean value):")
        display(filtered_df[[numeric_field_id]].head())
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id}:\n")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Grouping
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
            print(f"Grouped by {group_field_id}:\n")
            display(grouped_df.head())
    else:
        print("No numeric field found in first record set for demonstration.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships using pandas and matplotlib/seaborn. Example: distribution histogram for the selected numeric field.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_field_id and numeric_field_id in dataframes[record_set_ids[0]].columns:
    plt.figure(figsize=(8,5))
    sns.histplot(dataframes[record_set_ids[0]][numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
This notebook has demonstrated loading, schema exploration, and basic record processing of the FAIR^2 dataset using only Croissant `@id` for all data structure references. You can adapt the workflow for your own analyses or further data cleaning, modeling, or visualization.
